# Brain CT Image Analysis
## Step 01 — Data loading, matching and visualization

In [ ]:

from pathlib import Path
import json, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pydicom
from matplotlib.colors import ListedColormap
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D

warnings.filterwarnings("ignore")

ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/mehdipaykanheyrati/iaaa-contest-bct/Data"),
    Path("/kaggle/input/iaaa-contest-bct/Data"),
]

ROOT = next((p for p in ROOT_CANDIDATES if p.exists()), None)
if ROOT is None:
    raise FileNotFoundError("Dataset root was not found.")

ANNOTATIONS_DIR = ROOT / "annotations"
TRAINING_DIR = ROOT / "training"
TRAINING_DF_PATH = ROOT / "training_df.pkl"

print("ROOT:", ROOT)
print("annotations:", ANNOTATIONS_DIR.exists())
print("training:", TRAINING_DIR.exists())
print("training_df:", TRAINING_DF_PATH.exists())


In [ ]:

training_df = pd.read_pickle(TRAINING_DF_PATH).copy()

def normalize_series_id(x):
    if pd.isna(x):
        return ""
    try:
        return str(int(float(x)))
    except Exception:
        return str(x).strip()

training_df["_SeriesID_str"] = training_df["dicom_series.id"].map(normalize_series_id)
training_df["_SOP_str"] = training_df["dicom_series.SOPInstanceUID"].astype(str)

print("training_df shape:", training_df.shape)
print("number of series:", training_df["_SeriesID_str"].nunique())
display(training_df.head())


In [ ]:

ICH_CLASS_NAMES = {1:"IVH", 2:"IPH", 3:"SDH", 4:"EDH", 5:"SAH"}
ICH_COLORS = {1:"#ff4d4d", 2:"#ffb000", 3:"#00a8ff", 4:"#d64dff", 5:"#38d66b"}

def decode_segmentation_rle(segmentation_rle):
    if segmentation_rle is None:
        return None

    shape = tuple(int(x) for x in segmentation_rle["shape"])
    counts = segmentation_rle["counts"]

    if len(counts) % 2 != 0:
        raise ValueError("Invalid RLE")

    flat = np.empty(int(np.prod(shape)), dtype=np.uint8)
    pos = 0

    for value, run_len in zip(counts[0::2], counts[1::2]):
        run_len = int(run_len)
        flat[pos:pos+run_len] = int(value)
        pos += run_len

    if pos != flat.size:
        raise ValueError(f"Decoded {pos} pixels, expected {flat.size}")

    return flat.reshape(shape, order="C")


def load_json_for_slice(series_id, sop_uid):
    path = ANNOTATIONS_DIR / str(series_id) / f"{sop_uid}.json"
    if not path.exists():
        return None, None

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    return data, path


In [ ]:

def dicom_slice_position(ds):
    try:
        iop = np.asarray(ds.ImageOrientationPatient, dtype=float)
        ipp = np.asarray(ds.ImagePositionPatient, dtype=float)

        row_cosine = iop[:3]
        col_cosine = iop[3:]
        normal = np.cross(row_cosine, col_cosine)

        return float(np.dot(ipp, normal))
    except Exception:
        return None


def dicom_sort_key(ds):
    pos = dicom_slice_position(ds)
    if pos is not None:
        return (0, pos)

    try:
        return (1, float(ds.InstanceNumber))
    except Exception:
        return (2, 0)


def dicom_to_hu(ds):
    image = ds.pixel_array.astype(np.float32)
    slope = float(getattr(ds, "RescaleSlope", 1.0))
    intercept = float(getattr(ds, "RescaleIntercept", 0.0))
    return image * slope + intercept


def apply_window(image_hu, center=40, width=80):
    low = center - width/2
    high = center + width/2
    return np.clip(image_hu, low, high)


In [ ]:

def load_ct_series(series_id):
    sid = normalize_series_id(series_id)
    series_dir = TRAINING_DIR / sid

    if not series_dir.exists():
        raise FileNotFoundError(f"Series folder not found: {series_dir}")

    dicom_items = []

    for file_path in series_dir.rglob("*"):
        if not file_path.is_file():
            continue

        try:
            ds = pydicom.dcmread(str(file_path), force=True)
            if not hasattr(ds, "PixelData"):
                continue

            dicom_items.append({"path": file_path, "ds": ds})
        except Exception:
            continue

    if len(dicom_items) == 0:
        raise RuntimeError(f"No readable DICOM images found in {series_dir}")

    dicom_items.sort(key=lambda x: dicom_sort_key(x["ds"]))

    series_df = training_df[training_df["_SeriesID_str"] == sid].copy()

    metadata_by_sop = {
        str(row["_SOP_str"]): row
        for _, row in series_df.iterrows()
    }

    slices = []

    for i, item in enumerate(dicom_items, start=1):
        ds = item["ds"]
        sop_uid = str(getattr(ds, "SOPInstanceUID", ""))
        hu = dicom_to_hu(ds)

        json_data, json_path = load_json_for_slice(sid, sop_uid)

        mask = None
        if json_data is not None and "segmentation_rle" in json_data:
            mask = decode_segmentation_rle(json_data["segmentation_rle"])

        metadata_row = metadata_by_sop.get(sop_uid)

        slices.append({
            "slice_number": i,
            "dicom_path": item["path"],
            "dicom": ds,
            "sop_uid": sop_uid,
            "hu": hu,
            "metadata": metadata_row,
            "json_exists": json_data is not None,
            "json_path": json_path,
            "json": json_data,
            "mask": mask,
        })

    triage_values = []
    if "triage_class" in series_df.columns:
        triage_values = (
            series_df["triage_class"].dropna().astype(int).unique().tolist()
        )

    subtype_columns = {
        "IVH":"IntraventricularHemorrhage",
        "IPH":"IntraparenchymalHemorrhage",
        "SDH":"SubduralHemorrhage",
        "EDH":"EpiduralHemorrhage",
        "SAH":"SubarachnoidHemorrhage",
    }

    positive_subtypes = []
    for name, col in subtype_columns.items():
        if col in series_df.columns and series_df[col].fillna(False).astype(bool).any():
            positive_subtypes.append(name)

    fracture_present = False
    if "SkullFracture" in series_df.columns:
        fracture_present = series_df["SkullFracture"].fillna(False).astype(bool).any()

    mls_max = np.nan
    if "MidlineShiftMM" in series_df.columns:
        vals = pd.to_numeric(series_df["MidlineShiftMM"], errors="coerce").dropna()
        if len(vals):
            mls_max = float(vals.max())

    summary = {
        "Series_ID": sid,
        "Actual_DICOM_Count": len(slices),
        "training_df_rows": len(series_df),
        "JSON_Count": sum(s["json_exists"] for s in slices),
        "Slices_without_JSON": sum(not s["json_exists"] for s in slices),
        "Positive_ICH_Subtypes": positive_subtypes,
        "Fracture_present": fracture_present,
        "Max_MidlineShiftMM_in_training_df": mls_max,
        "triage_class": triage_values[0] if len(triage_values) == 1 else triage_values,
    }

    return {
        "series_id": sid,
        "series_dir": series_dir,
        "series_df": series_df,
        "slices": slices,
        "summary": summary,
    }


In [ ]:

def print_series_summary(case):
    print("=" * 60)
    print(f"SERIES {case['series_id']}")
    print("=" * 60)
    for key, value in case["summary"].items():
        print(f"{key:35s}: {value}")
    print("=" * 60)


def get_positive_slice_numbers(case):
    positive = []
    for s in case["slices"]:
        if s["mask"] is not None and np.any(s["mask"] > 0):
            positive.append(s["slice_number"])
    return positive


In [ ]:

def get_slice_ich_labels(metadata_row):
    if metadata_row is None:
        return []

    cols = {
        "IVH":"IntraventricularHemorrhage",
        "IPH":"IntraparenchymalHemorrhage",
        "SDH":"SubduralHemorrhage",
        "EDH":"EpiduralHemorrhage",
        "SAH":"SubarachnoidHemorrhage",
    }

    labels = []
    for short, col in cols.items():
        try:
            if bool(metadata_row[col]):
                labels.append(short)
        except Exception:
            pass
    return labels


def show_series(
    case,
    slice_numbers=None,
    n=9,
    window_center=40,
    window_width=80,
    show_mask=True,
    show_keypoints=True,
    show_fracture_boxes=True,
    figsize_per_image=4,
):
    slices = case["slices"]
    total = len(slices)

    if slice_numbers is None:
        count = min(n, total)
        indices = np.linspace(0, total-1, count, dtype=int).tolist()
    else:
        indices = [int(x)-1 for x in slice_numbers if 1 <= int(x) <= total]

    if not indices:
        print("No valid slices selected.")
        return

    ncols = min(3, len(indices))
    nrows = math.ceil(len(indices)/ncols)

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(figsize_per_image*ncols, figsize_per_image*nrows)
    )
    axes = np.atleast_1d(axes).ravel()

    cmap = ListedColormap([
        ICH_COLORS[1], ICH_COLORS[2], ICH_COLORS[3],
        ICH_COLORS[4], ICH_COLORS[5]
    ])

    for ax_i, slice_index in enumerate(indices):
        ax = axes[ax_i]
        s = slices[slice_index]

        image = apply_window(s["hu"], center=window_center, width=window_width)
        ax.imshow(image, cmap="gray", origin="upper")

        if show_mask and s["mask"] is not None:
            masked = np.ma.masked_where(s["mask"] == 0, s["mask"])
            ax.imshow(masked, cmap=cmap, vmin=1, vmax=5, alpha=0.40, origin="upper")

        json_data = s["json"] or {}

        if show_keypoints:
            keypoints = json_data.get("keypoints") or {}
            for name, xy in keypoints.items():
                if xy is None or len(xy) < 2:
                    continue
                x, y = float(xy[0]), float(xy[1])
                ax.scatter(x, y, s=30, marker="x", linewidths=2)
                ax.text(x+5, y-5, name, fontsize=7)

        if show_fracture_boxes:
            boxes = json_data.get("boxes_xywh") or []
            for box in boxes:
                if len(box) < 4:
                    continue
                x, y, w, h = map(float, box[:4])
                ax.add_patch(Rectangle((x,y), w, h, fill=False, linewidth=2))

        metadata = s["metadata"]
        labels = get_slice_ich_labels(metadata)
        label_text = ",".join(labels) if labels else "No ICH label"

        mls = ""
        fracture = ""

        if metadata is not None:
            try:
                val = metadata["MidlineShiftMM"]
                if pd.notna(val):
                    mls = f" | MLS={float(val):.2f} mm"
            except Exception:
                pass

            try:
                fracture = f" | Fx={bool(metadata['SkullFracture'])}"
            except Exception:
                pass

        json_status = "JSON" if s["json_exists"] else "NO JSON"

        ax.set_title(
            f"Slice {s['slice_number']}/{total}\n"
            f"{label_text} | {json_status}{mls}{fracture}",
            fontsize=9
        )
        ax.axis("off")

    for j in range(len(indices), len(axes)):
        axes[j].axis("off")

    if show_mask:
        legend_items = [
            Line2D([0],[0], marker="s", linestyle="None", markersize=10,
                   markerfacecolor=ICH_COLORS[k],
                   label=f"{k}: {ICH_CLASS_NAMES[k]}")
            for k in range(1,6)
        ]
        fig.legend(handles=legend_items, loc="lower center", ncol=5, fontsize=9)

    fig.suptitle(
        f"Series {case['series_id']} — {total} actual DICOM slices",
        fontsize=14
    )

    plt.tight_layout(rect=[0,0.06,1,0.95])
    plt.show()


In [ ]:

case_447 = load_ct_series(447)

print_series_summary(case_447)

positive_slices = get_positive_slice_numbers(case_447)
print("Slices with non-zero segmentation:", positive_slices)

show_series(
    case_447,
    n=9,
    window_center=40,
    window_width=80,
)


In [ ]:

# Show only slices with segmentation annotations
if len(positive_slices) > 0:
    selected = positive_slices[:12]

    show_series(
        case_447,
        slice_numbers=selected,
        window_center=40,
        window_width=80,
        show_mask=True,
        show_keypoints=True,
        show_fracture_boxes=True,
    )
else:
    print("No positive segmentation found.")


In [ ]:

def inspect_slice(case, slice_number):
    s = case["slices"][slice_number - 1]

    print("=" * 80)
    print("Series:", case["series_id"])
    print("Slice:", slice_number, "/", len(case["slices"]))
    print("DICOM:", s["dicom_path"])
    print("SOPInstanceUID:", s["sop_uid"])
    print("JSON exists:", s["json_exists"])
    print("JSON path:", s["json_path"])
    print("=" * 80)

    if s["metadata"] is not None:
        display(
            s["metadata"]
            .drop(labels=["_SeriesID_str", "_SOP_str"], errors="ignore")
            .to_frame("value")
        )
    else:
        print("No matching training_df row for this DICOM slice.")

    show_series(
        case,
        slice_numbers=[slice_number],
        window_center=40,
        window_width=80,
        show_mask=True,
        show_keypoints=True,
        show_fracture_boxes=True,
        figsize_per_image=7,
    )

# Example:
# inspect_slice(case_447, 15)
